In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

# FLenQA output sanity check

Check the completed `flenqa_full_run.ipynb` artifact without loading the model or lens. Whole-run checks are cheap; detailed rank checks use three deterministic prompts.

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=False)
context

## 1. Check the whole run

Verify the three table families, schemas, shard names, and row counts. Parquet metadata supplies the counts without loading the large top-k table.

In [ ]:
import math

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import transformers

from experiments.jlens_readout_sanity.constants import MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.storage import (
    REQUIRED_TABLES,
    TABLE_SCHEMAS,
)

RUN_DIR = context.runs_dir / "flenqa-full-run"
TOP_K = 25
shards = {
    table: sorted((RUN_DIR / table).glob("shard-*.parquet"))
    for table in REQUIRED_TABLES
}
assert all(shards.values()), f"Missing FLenQA shards under {RUN_DIR}"
assert len({tuple(path.name for path in paths) for paths in shards.values()}) == 1
for table, paths in shards.items():
    assert all(pq.read_schema(path).equals(TABLE_SCHEMAS[table]) for path in paths)

shard_row_counts = {
    table: [pq.ParquetFile(path).metadata.num_rows for path in paths]
    for table, paths in shards.items()
}
assert all(count > 0 for counts in shard_row_counts.values() for count in counts)
row_counts = {table: sum(counts) for table, counts in shard_row_counts.items()}
assert row_counts["prompts"] == 9_862
assert row_counts["positions"] > 0 and row_counts["topk"] > 0
{"shards_per_table": len(shards["prompts"]), **row_counts}

## 2. Check prompt and position keys

Stream the prompt IDs and token lengths, then make sure every saved position belongs to a prompt and falls inside its token sequence.

In [ ]:
prompt_data = ds.dataset(shards["prompts"], format="parquet")
prompt_ids, canonical_indices, input_lengths = [], [], {}
for batch in prompt_data.to_batches(
    columns=["prompt_id", "canonical_index", "input_ids", "max_abs_logit_diff"]
):
    ids = batch["prompt_id"].to_pylist()
    lengths = pc.list_value_length(batch["input_ids"]).to_pylist()
    prompt_ids.extend(ids)
    canonical_indices.extend(batch["canonical_index"].to_pylist())
    input_lengths.update(zip(ids, lengths, strict=True))
    assert all(
        math.isfinite(value) and value >= 0
        for value in batch["max_abs_logit_diff"].to_pylist()
    )

assert len(set(prompt_ids)) == len(prompt_ids) == 9_862
assert canonical_indices == list(range(9_862))

position_data = ds.dataset(shards["positions"], format="parquet")
position_prompt_ids, execution_positions = set(), set()
for batch in position_data.to_batches(columns=["prompt_id", "position"]):
    for prompt_id, position in zip(
        batch["prompt_id"].to_pylist(),
        batch["position"].to_pylist(),
        strict=True,
    ):
        assert 0 <= position < input_lengths[prompt_id]
        position_prompt_ids.add(prompt_id)
        execution_positions.add((prompt_id, position))
assert position_prompt_ids == set(prompt_ids)

## 3. Inspect three prompts in detail

For the first, middle, and last prompts, require paired Jacobian/Logit groups, ranks 1–25, unique token IDs, and descending finite logits.

In [ ]:
import math
from collections import defaultdict


def check_topk_sample(rows, expected_positions, *, top_k):
    groups = defaultdict(list)
    for row in rows:
        assert (row["prompt_id"], row["position"]) in expected_positions
        assert math.isfinite(row["logit"])
        key = (
            row["prompt_id"],
            row["layer"],
            row["position"],
            row["lens_kind"],
        )
        groups[key].append(row)

    lens_kinds = defaultdict(set)
    layer_sets = defaultdict(set)
    for prompt_id, layer, position, lens_kind in groups:
        lens_kinds[prompt_id, layer, position].add(lens_kind)
        layer_sets[prompt_id, position].add(layer)
    base_groups = set(lens_kinds)
    assert {
        (prompt_id, position) for prompt_id, _, position in base_groups
    } == expected_positions
    assert all(kinds == {"jacobian", "logit"} for kinds in lens_kinds.values())
    assert len({tuple(sorted(layers)) for layers in layer_sets.values()}) == 1

    for group in groups.values():
        ranked = sorted(group, key=lambda row: row["rank"])
        assert [row["rank"] for row in ranked] == list(range(1, top_k + 1))
        assert len({row["token_id"] for row in ranked}) == top_k
        logits = [row["logit"] for row in ranked]
        assert all(
            left >= right for left, right in zip(logits, logits[1:], strict=False)
        )

    return len(base_groups)

In [ ]:
sample_ids = [prompt_ids[index] for index in (0, len(prompt_ids) // 2, -1)]
sample_filter = ds.field("prompt_id").isin(sample_ids)
sample_positions = position_data.to_table(filter=sample_filter).to_pylist()
expected_positions = {(row["prompt_id"], row["position"]) for row in sample_positions}
topk_data = ds.dataset(shards["topk"], format="parquet")
sample_topk = topk_data.to_table(filter=sample_filter).to_pylist()
checked_groups = check_topk_sample(sample_topk, expected_positions, top_k=TOP_K)
sample_layers = {row["layer"] for row in sample_topk}
assert row_counts["topk"] == len(execution_positions) * len(sample_layers) * 2 * TOP_K

tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
first_id = sample_ids[0]
final_position = next(
    row["position"]
    for row in sample_positions
    if row["prompt_id"] == first_id and row["label"] == "final_prompt"
)
final_layer = max(row["layer"] for row in sample_topk)
comparison = [
    {
        "lens": row["lens_kind"],
        "rank": row["rank"],
        "token": repr(tokenizer.decode([row["token_id"]])),
        "logit": row["logit"],
    }
    for row in sample_topk
    if row["prompt_id"] == first_id
    and row["position"] == final_position
    and row["layer"] == final_layer
    and row["rank"] <= 5
]
display(
    pa.Table.from_pylist(comparison).sort_by(
        [("lens", "ascending"), ("rank", "ascending")]
    )
)
print(
    f"PASS: {row_counts['prompts']:,} prompts; "
    f"{checked_groups:,} sampled prompt/layer/position groups checked."
)